# Part 3 - Machine Learning: predict no-shows and choose the model honestly
**Project: CareLoop Clinics - No-Show Guard**

**Goal:** a risk score for every booking that the front desk can use **the evening before the
appointment** to decide whom to phone. At that moment everything we use as a feature is already
known: the patient's age, distance, fee, how long ago they booked, how many times they missed
before, and whether today's reminder was sent.

Only about 1 booking in 5 is a no-show, so accuracy alone is misleading. We judge every model
by **recall**, **precision** and **ROC-AUC** as well.

**The rule for this whole notebook:** the test set is for *reporting only*. We never use it to
choose a model, a setting or a threshold. All choices are made on the training set with
cross-validation.

| Step | What we do |
|---|---|
| 3.1 | Build the features, split 80/20, compute the baseline |
| 3.2 | Logistic regression (with scaling) and what its coefficients mean |
| 3.3 | Decision tree: unlimited depth vs depth 3 (overfitting) |
| 3.4 | Random forest and a comparison table |
| 3.5 | Tune, choose the model and the threshold with cross-validation, then test once |

In [1]:
# Install the libraries this notebook needs (safe to re-run).
%pip install -q --disable-pip-version-check pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# On Google Colab the project files are not there yet, so we download (clone) the repo first.
if "google.colab" in str(get_ipython()) and not Path("MASAI-CAPSTONE-PROJ").exists():
    os.system("git clone -q https://github.com/Advay4706/MASAI-CAPSTONE-PROJ")
if Path("MASAI-CAPSTONE-PROJ").exists():
    os.chdir("MASAI-CAPSTONE-PROJ")

BASE = Path(".") if Path("data").exists() else Path("..")   # the project's top folder
OUT = BASE / "outputs"

RANDOM_STATE = 42     # the same "seed" everywhere, so every run gives the same result
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.8.0 | pandas 3.0.1 | numpy 2.4.3


## Step 3.1 - Features, split and baseline

**What:** turn the clean table into a feature table `X` (numbers only) and a target `y`
(`no_show`, 1 or 0). Split the rows 80% for training and 20% for testing. Work out the
*baseline*: the accuracy of a "model" that always says "Attended".

**Why each choice:**

- **Features used:** `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`, `reminder_sent`,
  `booking_channel`, `visit_type`.
- **Not used:** `appointment_id` and `patient_id` (labels, not information), the dates (already
  summarised by `lead_days`), `clinic_id` and `gender` (left out by the brief), and `status`
  (it *is* the answer - using it would be cheating, called **target leakage**).
- **`.map({"Yes": 1, "No": 0})`** turns a two-value text column into 1/0.
- **One-hot encoding** (`pd.get_dummies`) turns a text column with several values into one
  1/0 column per value. We cannot write App=1, Phone=2, Walk-In=3, because that would tell the
  model that Walk-In is "three times" App, which means nothing.
- **Train/test split:** the model learns from the training rows and is graded on test rows it
  has never seen. `random_state=42` makes the random split repeatable.
- We do **not** re-sort the rows: the split depends on the row order of the file.

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(OUT / "appointments_clean.csv")        # keep the file's row order
print("Loaded:", df.shape)

feature_columns = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
                   "reminder_sent", "booking_channel", "visit_type"]

X = df[feature_columns].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})          # text -> 1/0
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])          # one-hot encoding
y = df["no_show"]                                                         # the target

print("X has", X.shape[1], "columns:")
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print("\nTraining rows:", len(X_train), "| test rows:", len(X_test))
print("No-shows in the test set:", int(y_test.sum()), f"({y_test.mean():.1%})")
print("No-shows in the training set:", int(y_train.sum()), f"({y_train.mean():.1%})")

# Baseline: always predict "Attended" (0). It is right for every patient who attended.
baseline_accuracy = (y_test == 0).mean()
print(f"\nBaseline accuracy (always predict 'Attended'): {baseline_accuracy:.4f} "
      f"= {(y_test == 0).sum()} of {len(y_test)}")

Loaded: (990, 16)
X has 11 columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training rows: 792 | test rows: 198
No-shows in the test set: 49 (24.7%)
No-shows in the training set: 169 (21.3%)

Baseline accuracy (always predict 'Attended'): 0.7525 = 149 of 198


**Reading the result**

- `X` has **11 columns**: 6 numeric ones, 3 for `booking_channel` and 2 for `visit_type`.
- **792** training rows and **198** test rows; **49** of the test rows are no-shows.
- **Baseline accuracy is 0.7525** (149 of 198). A model that never flags anyone is 75% "accurate"
  and completely useless - it catches zero no-shows. This is why accuracy alone is a trap on
  **imbalanced** data, and why a real model must beat 0.7525 *and* have useful recall.

## Step 3.2 - Logistic regression

**What:** scale the features, fit a logistic regression, and report its test metrics and
coefficients.

**The idea:** logistic regression adds up `coefficient x feature` for all features and squeezes
the total through an S-shaped curve to get a probability between 0 and 1. A positive
coefficient pushes the risk up, a negative one pushes it down.

**Why scale:** `fee_inr` is in the hundreds, `reminder_sent` is 0 or 1. `StandardScaler` puts
every feature on the same footing (mean 0, standard deviation 1), which helps the fitting and
makes the coefficients comparable in size.

**Fit the scaler on the training set only.** The test set stands for future, unseen data. If
its mean and standard deviation leaked into the scaler, the test would no longer be honest
(**data leakage**). So: `fit_transform` on train, only `transform` on test.

**The metrics** (a "positive" is a no-show):

| Metric | Question it answers |
|---|---|
| Accuracy | Of all predictions, how many were right? |
| Precision | Of the patients we flagged, how many really missed? |
| Recall | Of the patients who really missed, how many did we flag? |
| F1 | One number that balances precision and recall |
| ROC-AUC | If we pick one no-show and one attender at random, how often does the no-show get the higher risk score? 0.5 = coin flip, 1.0 = perfect. It does not depend on a threshold. |

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)     # learn mean/std from TRAIN, then scale train
X_test_s = scaler.transform(X_test)           # scale test with the TRAIN mean/std

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

lr_pred = log_reg.predict(X_test_s)                    # 0/1 using the default 0.5 threshold
lr_prob = log_reg.predict_proba(X_test_s)[:, 1]        # probability of class 1 (no-show)

results = {}      # we collect every model's test metrics here for the comparison table
results["Logistic regression"] = {
    "accuracy": accuracy_score(y_test, lr_pred),
    "recall": recall_score(y_test, lr_pred),
    "roc_auc": roc_auc_score(y_test, lr_prob),
}

print(f"Accuracy : {accuracy_score(y_test, lr_pred):.3f}   (baseline {baseline_accuracy:.4f})")
print(f"Precision: {precision_score(y_test, lr_pred):.3f}")
print(f"Recall   : {recall_score(y_test, lr_pred):.3f}")
print(f"F1       : {f1_score(y_test, lr_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, lr_prob):.3f}")

cm = confusion_matrix(y_test, lr_pred)
print("\nConfusion matrix (rows = actual, columns = predicted):")
display(pd.DataFrame(cm, index=["actual Attended", "actual No-Show"],
                     columns=["predicted Attended", "predicted No-Show"]))

Accuracy : 0.788   (baseline 0.7525)
Precision: 0.630
Recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789

Confusion matrix (rows = actual, columns = predicted):


,predicted Attended,predicted No-Show
actual Attended,139,10
actual No-Show,32,17


### Coefficients
One coefficient per feature, in the column order of `X`. Because the features were scaled,
each coefficient is the effect of moving that feature by **one standard deviation**.

In [5]:
coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": log_reg.coef_[0],
    "feature_std (scaler.scale_)": scaler.scale_,
})
display(coef_table.round(3))

i = list(X.columns).index("reminder_sent")
coef_reminder = log_reg.coef_[0][i]
std_reminder = scaler.scale_[i]

odds_per_std = np.exp(coef_reminder)                    # effect of +1 standard deviation
odds_no_to_yes = np.exp(coef_reminder / std_reminder)   # effect of the real change No (0) -> Yes (1)

print(f"reminder_sent coefficient (scaled feature): {coef_reminder:.3f}")
print(f"standard deviation of reminder_sent        : {std_reminder:.3f}")
print(f"np.exp(coef)         = {odds_per_std:.3f}  -> odds x {odds_per_std:.2f} per ONE STANDARD DEVIATION")
print(f"np.exp(coef / scale) = {odds_no_to_yes:.3f}  -> odds x {odds_no_to_yes:.2f} for the real change No -> Yes "
      f"({1 - odds_no_to_yes:.0%} lower odds)")

,feature,coefficient,feature_std (scaler.scale_)
0,age,-0.254,16.656
1,distance_km,0.253,6.052
2,fee_inr,0.453,184.254
3,lead_days,0.614,8.614
4,prior_no_shows,0.650,0.969
5,reminder_sent,-0.581,0.495
6,booking_channel_App,-0.047,0.486
7,booking_channel_Phone,0.216,0.481
8,booking_channel_Walk-In,-0.186,0.435
9,visit_type_Consultation,0.016,0.493


reminder_sent coefficient (scaled feature): -0.581
standard deviation of reminder_sent        : 0.495
np.exp(coef)         = 0.560  -> odds x 0.56 per ONE STANDARD DEVIATION
np.exp(coef / scale) = 0.309  -> odds x 0.31 for the real change No -> Yes (69% lower odds)


**Reading the result**

- Accuracy **0.788** beats the 0.7525 baseline, and ROC-AUC is **0.789** - the model ranks
  patients by risk clearly better than chance.
- But at the default 0.5 threshold, **recall is only 0.347**: it catches 17 of the 49 no-shows
  and misses 32 (confusion matrix `[[139, 10], [32, 17]]`). Precision is 0.630 (17 of the 27
  flagged really missed), F1 is 0.447. We fix the low recall with a better threshold in Step 3.5.
- `reminder_sent` has the most negative coefficient (**-0.581**). `prior_no_shows` (+0.650)
  and `lead_days` (+0.614) are the strongest positive ones - the same lead-time story that
  the SQL bands showed in Part 2.

**The two numbers for `reminder_sent`, and why they differ**

- `np.exp(coef)` = **0.56**. This is the change in odds for a one-*standard-deviation* step in
  the scaled feature. A reminder is not "one standard deviation"; its standard deviation is 0.495.
- `np.exp(coef / scale)` = **0.31**. Dividing by the standard deviation converts the coefficient
  back to the feature's real units, so this is the change in odds for the real step from
  No (0) to Yes (1).
- **They differ because the model was fitted on scaled features: going from No to Yes is about
  two standard deviations (1 / 0.495), not one.** Reporting 0.56 ("44% lower") would understate it.

**Conclusion, worded carefully:** in this model, holding the other features constant, appointments
with a reminder have about **69% lower odds** of a no-show than appointments without one. This is
an **association, not proof of cause**: the data are observational. Reminders were not handed out
at random - maybe app users, who are more organised anyway, get more reminders. Only an experiment
(send reminders to a random half) could prove that reminders *cause* fewer no-shows.

(Odds are not probability: odds = p / (1 - p). "69% lower odds" does not mean "69% fewer no-shows".)